In [ ]:
# =====================================================================
# BOLUM 0 - Metodolojik duzeltme: ayarlar, veri ve sabit yapilar
# Bu kod optimizasyonu YENIDEN CALISTIRMAZ (zorunlu hâller disinda).
# Hiperparametre araliklari DEGISTIRILMEZ.
# =====================================================================
import os
import re
import ast
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from joblib import Parallel, delayed

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

RANDOM_STATE = 42
MAIN_PSO_SEED = 42          # ana karsilastirmada KULLANILACAK TEK seed
N_SPLITS = 5
ROUND_DEC = 6
EPS = 1e-8
N_JOBS = -1
HIGH_FREQ_THRESHOLD = 2.5

# Geri kazanim basarisiz olursa yalnizca eksik kombinasyon icin PSO tekrar calisir
ALLOW_PSO_RERUN_IF_MISSING = True
PSO_SWARM_SIZE = 10
PSO_N_ITERATIONS = 10
PSO_W_START, PSO_W_END = 0.9, 0.4
PSO_C1, PSO_C2 = 2.0, 2.0
PSO_VELOCITY_RATIO = 0.5
PSO_COUNT_INITIAL_AS_ITERATION = True

# ---- Portable project paths ---------------------------------------------
from pathlib import Path

def find_project_root():
    """Locate the repository root from the current working directory."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "masonry_tower_primary_dataset.xlsx").is_file():
            return candidate
    raise FileNotFoundError(
        "Project root could not be located. Run this notebook from the repository "
        "root or from its code/ directory, and keep the data/ directory unchanged."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAT_FILE = DATA_DIR / "masonry_tower_primary_dataset.xlsx"
# ------------------------------------------------------------------------
SPLIT_FILE = os.path.join(OUTPUT_DIR, "Data_Split_Assignment.xlsx")
MODEL_DIR  = os.path.join(OUTPUT_DIR, "Models")
os.makedirs(MODEL_DIR, exist_ok=True)

RS_SONUC_FILE   = os.path.join(OUTPUT_DIR, "Randomized_Search_Results.xlsx")
PSO_SONUC_FILE  = os.path.join(OUTPUT_DIR, "PSO_Search_Results.xlsx")
PSO_KARAR_FILE  = os.path.join(OUTPUT_DIR, "PSO_Stability_Results.xlsx")
ESKI_COST_FILE  = os.path.join(OUTPUT_DIR, "Optimization_Computational_Cost.xlsx")
ESKI_PARAM_FILE = os.path.join(OUTPUT_DIR, "Optimized_Model_Parameters.xlsx")

AVP_DIR   = os.path.join(OUTPUT_DIR, "Optimized_Actual_vs_Predicted")
RESID_DIR = os.path.join(OUTPUT_DIR, "Optimized_Residual_Plots")
COMP_DIR  = os.path.join(OUTPUT_DIR, "Optimization_Comparison")
for k in [AVP_DIR, RESID_DIR, COMP_DIR]:
    os.makedirs(k, exist_ok=True)

GEO_COLS = ["Height (m)", "Section a (m)", "Section b (m)", "Wall Thickness (m)",
            "Opening z/H", "Opening Ratio x (%)", "Opening Ratio y (%)"]
MAT_COLS = ["E (MPa)", "d (kg/m3)"]
FEATURE_COLS = GEO_COLS + MAT_COLS
TARGETS = ["f1 (Hz)", "f2 (Hz)"]
TARGET_KISA = {"f1 (Hz)": "f1", "f2 (Hz)": "f2"}
KISA_TARGET = {v: k for k, v in TARGET_KISA.items()}
MODELLER = ["SVR", "GBR"]
YONTEMLER = ["Baseline", "Randomized Search", "PSO"]
BEKLENEN_DISARIDA = {"G170", "G187", "G183", "G068", "G099", "G076"}

DPI = 300
sns.set_style("white")
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.titlesize": 12, "axes.labelsize": 12,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 9.5,
    "axes.linewidth": 0.9, "savefig.dpi": DPI, "savefig.bbox": "tight",
})
RENK = {"Baseline": "#8C8C8C", "Randomized Search": "#4C72B0", "PSO": "#55A868"}
RENK_VURGU = "#C44E52"
HEDEF_ETIKET = {"f1": "$f_1$", "f2": "$f_2$"}


def kaydet(fig, klasor, ad):
    """Figuru PNG olarak kaydeder ve bellekten temizler."""
    yol = os.path.join(klasor, ad + ".png")
    fig.savefig(yol, dpi=DPI)
    plt.close(fig)
    print("Guncellendi:", yol)


# ---- Veri ve dondurulmus bolme (Asama 3 ile birebir ayni) -------------
def geometry_id_olustur(veri, geo_cols=GEO_COLS, ndec=ROUND_DEC):
    """Onceki asamalarla birebir ayni Geometry_ID uretimi."""
    anahtar = veri[geo_cols].round(ndec).astype(str).agg("|".join, axis=1)
    esleme = {k: f"G{i+1:03d}" for i, k in enumerate(pd.unique(anahtar))}
    return anahtar.map(esleme)


def dondurulmus_bolmeyi_yukle(veri):
    """Data_Split_Assignment.xlsx dosyasindaki Split ve CV_Fold atamalarini okur."""
    if not os.path.exists(SPLIT_FILE):
        raise FileNotFoundError(f"Bolme dosyasi bulunamadi: {SPLIT_FILE}")
    atama = pd.read_excel(SPLIT_FILE, sheet_name="Row_assignment").sort_values("Row_index")
    if len(atama) != len(veri):
        raise ValueError(f"Satir sayisi uyusmuyor: veri={len(veri)}, bolme={len(atama)}")
    if not (atama["Geometry_ID"].values == veri["Geometry_ID"].values).all():
        raise ValueError("Geometry_ID sirasi bolme dosyasiyla uyusmuyor.")
    veri = veri.copy()
    veri["Split"] = atama["Split"].values
    veri["CV_Fold"] = atama["CV_Fold"].values

    for ad, bek, goz in [("Train records", 912, int((veri["Split"] == "Train").sum())),
                         ("Test records", 234, int((veri["Split"] == "Test").sum())),
                         ("Train geometries", 152,
                          veri.loc[veri["Split"] == "Train", "Geometry_ID"].nunique()),
                         ("Test geometries", 39,
                          veri.loc[veri["Split"] == "Test", "Geometry_ID"].nunique())]:
        if bek != goz:
            raise ValueError(f"Bolme yapisi farkli -> {ad}: beklenen {bek}, gozlenen {goz}")

    ortak = (set(veri.loc[veri["Split"] == "Train", "Geometry_ID"]) &
             set(veri.loc[veri["Split"] == "Test", "Geometry_ID"]))
    if ortak:
        raise ValueError(f"VERI SIZINTISI: {len(ortak)} geometri hem egitimde hem testte.")
    print("Dondurulmus bolme dogrulandi (912/234 kayit, 152/39 geometri).")
    return veri


df = pd.read_excel(MAT_FILE, sheet_name=0)
df["Geometry_ID"] = geometry_id_olustur(df)
df = dondurulmus_bolmeyi_yukle(df)

train_mask = (df["Split"] == "Train").values
test_mask = (df["Split"] == "Test").values
X_train = df.loc[train_mask, FEATURE_COLS].reset_index(drop=True)
X_test = df.loc[test_mask, FEATURE_COLS].reset_index(drop=True)
geo_train = df.loc[train_mask, "Geometry_ID"].reset_index(drop=True)
geo_test = df.loc[test_mask, "Geometry_ID"].reset_index(drop=True)
fold_train = df.loc[train_mask, "CV_Fold"].astype(int).reset_index(drop=True)
y_train = {t: df.loc[train_mask, t].reset_index(drop=True) for t in TARGETS}
y_test = {t: df.loc[test_mask, t].reset_index(drop=True) for t in TARGETS}

# ---- Sabit CV katlari -------------------------------------------------
CV_SPLITS, kontrol_satirlari = [], []
for fold in range(1, N_SPLITS + 1):
    val_idx = np.where(fold_train.values == fold)[0]
    tr_idx = np.where(fold_train.values != fold)[0]
    ortak = set(geo_train.iloc[tr_idx]) & set(geo_train.iloc[val_idx])
    if len(ortak) != 0:
        raise ValueError(f"VERI SIZINTISI: Fold {fold} icinde {len(ortak)} ortak geometri.")
    CV_SPLITS.append((tr_idx, val_idx))
    kontrol_satirlari.append({"Fold": fold, "Train_samples": len(tr_idx),
                              "Validation_samples": len(val_idx),
                              "Train_geometries": geo_train.iloc[tr_idx].nunique(),
                              "Validation_geometries": geo_train.iloc[val_idx].nunique(),
                              "Common_geometries": 0})
CV_KONTROL = pd.DataFrame(kontrol_satirlari)
print("\nSabit CV katlari dogrulandi:")
print(CV_KONTROL.to_string(index=False))

# ---- Test alt grup maskeleri (model/seed seciminde KULLANILMAZ) -------
disarida_mask = np.zeros(len(X_test), dtype=bool)
for kol in FEATURE_COLS:
    disarida_mask |= (X_test[kol] < X_train[kol].min()).values
    disarida_mask |= (X_test[kol] > X_train[kol].max()).values
bulunan_disarida = set(geo_test[disarida_mask].unique())
if bulunan_disarida != BEKLENEN_DISARIDA:
    print("UYARI: Tasarim uzayi disi geometri kumesi beklenenden farkli!")
    print("  Bulunan :", sorted(bulunan_disarida))
yuksek_frekans_mask = {t: (y_test[t].values > HIGH_FREQ_THRESHOLD) for t in TARGETS}

In [ ]:
# =====================================================================
# BOLUM 1 - Randomized Search ve PSO seed 42 hiperparametrelerinin
#           mevcut sonuc dosyalarindan geri kazanilmasi
# =====================================================================

def parametre_metnini_coz(metin):
    """str(dict) bicimindeki parametre metnini sozluge cevirir.
    NumPy 2.x'in 'np.float64(...)' sarmalayicilarina karsi temizlik yapar."""
    if not isinstance(metin, str) or not metin.strip():
        return None
    temiz = re.sub(r"np\.(float64|float32|int64|int32|bool_)\(([^()]*)\)", r"\2", metin)
    temiz = temiz.replace("nan", "None")
    try:
        sozluk = ast.literal_eval(temiz)
        return sozluk if isinstance(sozluk, dict) else None
    except (ValueError, SyntaxError):
        return None


INT_PARAMS = {"model__n_estimators", "model__max_depth",
              "model__min_samples_split", "model__min_samples_leaf"}


def parametreleri_duzelt(parametreler):
    """Excel/metin donusumu sonrasi tip bozulmalarini giderir."""
    if not parametreler:
        return {}
    duzeltilmis = {}
    for anahtar, deger in parametreler.items():
        if anahtar in INT_PARAMS and deger is not None:
            duzeltilmis[anahtar] = int(round(float(deger)))
        elif isinstance(deger, float) and np.isnan(deger):
            duzeltilmis[anahtar] = None
        else:
            duzeltilmis[anahtar] = deger
    return duzeltilmis


def rs_parametrelerini_oku():
    """Randomized Search en iyi parametrelerini mevcut sonuc dosyasindan okur."""
    kaynaklar = [(RS_SONUC_FILE, "Best_parameters"), (ESKI_PARAM_FILE, "Best_parameters")]
    for dosya, sayfa in kaynaklar:
        if not os.path.exists(dosya):
            continue
        try:
            tablo = pd.read_excel(dosya, sheet_name=sayfa)
        except Exception:
            continue
        if "Optimization_method" in tablo.columns:
            tablo = tablo[tablo["Optimization_method"] == "Randomized Search"]
        sonuc = {}
        for _, satir in tablo.iterrows():
            sozluk = parametre_metnini_coz(satir.get("Best_parameters"))
            if sozluk is not None:
                sonuc[(satir["Model"], KISA_TARGET[satir["Target"]])] = parametreleri_duzelt(sozluk)
        if len(sonuc) == 4:
            print(f"Randomized Search parametreleri okundu: {os.path.basename(dosya)} / {sayfa}")
            return sonuc
    raise FileNotFoundError(
        "Randomized Search en iyi parametreleri hicbir kayittan okunamadi. "
        "Randomized_Search_Results.xlsx dosyasinin mevcut oldugundan emin olun.")


def pso_seed42_kararlilik_dosyasindan():
    """PSO_Stability_Results.xlsx -> Runs sayfasindan seed 42 parametrelerini alir."""
    if not os.path.exists(PSO_KARAR_FILE):
        return {}, pd.DataFrame()
    kosular = pd.read_excel(PSO_KARAR_FILE, sheet_name="Runs")
    seed42 = kosular[kosular["Seed"] == MAIN_PSO_SEED]
    sonuc = {}
    for _, satir in seed42.iterrows():
        sozluk = parametre_metnini_coz(satir.get("Best_parameters"))
        if sozluk is not None:
            sonuc[(satir["Model"], KISA_TARGET[satir["Target"]])] = parametreleri_duzelt(sozluk)
    return sonuc, seed42


def pso_seed42_parcacik_dosyasindan():
    """PSO_Search_Results.xlsx parcacik kayitlarindan seed 42'nin en iyi
    kombinasyonunu bagimsiz olarak yeniden kurar (capraz kontrol icin)."""
    if not os.path.exists(PSO_SONUC_FILE):
        return {}, {}
    sonuc, kayitli_rmse = {}, {}
    xls = pd.ExcelFile(PSO_SONUC_FILE)
    for hedef in TARGETS:
        for model_adi in MODELLER:
            sayfa = f"{TARGET_KISA[hedef]}_{model_adi}"
            if sayfa not in xls.sheet_names:
                continue
            tablo = pd.read_excel(PSO_SONUC_FILE, sheet_name=sayfa)
            tablo = tablo[tablo["Seed"] == MAIN_PSO_SEED]
            if tablo.empty:
                continue
            en_iyi = tablo.loc[tablo["Mean_CV_RMSE"].idxmin()]

            if model_adi == "SVR":
                gerekli = ["C", "gamma", "epsilon"]
            else:
                gerekli = ["n_estimators", "learning_rate", "max_depth", "min_samples_split",
                           "min_samples_leaf", "subsample", "max_features", "loss"]
            if not all(k in tablo.columns for k in gerekli):
                continue

            parametreler = {}
            for k in gerekli:
                deger = en_iyi[k]
                if k == "max_features" and (pd.isna(deger) or str(deger).lower() in ("none", "nan")):
                    deger = None
                elif isinstance(deger, str) and deger not in ("sqrt", "log2", "huber",
                                                              "squared_error", "ls"):
                    try:
                        deger = float(deger)
                    except ValueError:
                        pass
                parametreler[f"model__{k}"] = deger

            sonuc[(model_adi, hedef)] = parametreleri_duzelt(parametreler)
            kayitli_rmse[(model_adi, hedef)] = float(en_iyi["Mean_CV_RMSE"])
    return sonuc, kayitli_rmse


rs_en_iyi = rs_parametrelerini_oku()
pso_kararlilik, seed42_kosular = pso_seed42_kararlilik_dosyasindan()
pso_parcacik, pso_parcacik_rmse = pso_seed42_parcacik_dosyasindan()

# ---- Iki bagimsiz kaynagin karsilastirilmasi --------------------------
print("\n--- PSO seed 42 parametre geri kazanimi ---")
pso_en_iyi, geri_kazanim_notlari = {}, []
for hedef in TARGETS:
    for model_adi in MODELLER:
        anahtar = (model_adi, hedef)
        a = pso_kararlilik.get(anahtar)
        b = pso_parcacik.get(anahtar)

        if a is not None and b is not None:
            uyum = all(str(a.get(k)) == str(b.get(k)) for k in set(a) | set(b))
            durum = "Iki kaynak uyumlu" if uyum else "UYUMSUZ - kararlilik dosyasi esas alindi"
            secilen, kaynak = a, "PSO_Stability_Results.xlsx (Runs)"
        elif a is not None:
            durum, secilen, kaynak = "Yalnizca kararlilik dosyasi", a, "PSO_Stability_Results.xlsx"
        elif b is not None:
            durum, secilen, kaynak = "Yalnizca parcacik dosyasi", b, "PSO_Search_Results.xlsx"
        else:
            durum, secilen, kaynak = "BULUNAMADI", None, "-"

        pso_en_iyi[anahtar] = secilen
        geri_kazanim_notlari.append({"Target": TARGET_KISA[hedef], "Model": model_adi,
                                     "Recovery_status": durum, "Source": kaynak})
        print(f"  {TARGET_KISA[hedef]} | {model_adi:3s} | {durum}")

eksik_kombinasyonlar = [k for k, v in pso_en_iyi.items() if v is None]

In [ ]:
# =====================================================================
# BOLUM 2 - Amac fonksiyonu, seed 42 dogrulamasi ve zorunlu yeniden kosu
# =====================================================================

def pipeline_olustur(model_adi):
    """SVR icin StandardScaler -> SVR, GBR icin yalnizca model (Asama 5 ile ayni)."""
    if model_adi == "SVR":
        return Pipeline([("scaler", StandardScaler()),
                         ("model", SVR(kernel="rbf", shrinking=True, cache_size=1000))])
    if model_adi == "GBR":
        return Pipeline([("model", GradientBoostingRegressor(random_state=RANDOM_STATE))])
    raise ValueError(f"Bilinmeyen model: {model_adi}")


def baseline_parametreleri(model_adi):
    """Asama 4 temel ayarlari."""
    return {"model__C": 1.0, "model__gamma": "scale", "model__epsilon": 0.1} \
        if model_adi == "SVR" else {}


def _tek_fold_rmse(model_adi, hedef_degerleri, parametreler, tr_idx, va_idx):
    """Bir fold icin modeli egitir ve validation RMSE dondurur."""
    pipe = pipeline_olustur(model_adi)
    if parametreler:
        pipe.set_params(**parametreler)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pipe.fit(X_train.iloc[tr_idx], hedef_degerleri.iloc[tr_idx])
    return float(np.sqrt(mean_squared_error(hedef_degerleri.iloc[va_idx],
                                            pipe.predict(X_train.iloc[va_idx]))))


def ortalama_cv_rmse(model_adi, hedef, parametreler):
    """Amac fonksiyonu: bes sabit fold uzerinde ortalama validation RMSE."""
    degerler = Parallel(n_jobs=N_JOBS)(
        delayed(_tek_fold_rmse)(model_adi, y_train[hedef], parametreler, tr, va)
        for tr, va in CV_SPLITS)
    return float(np.mean(degerler))


# ---- Zorunlu hâlde kullanilacak PSO (Asama 5 ile ayni ayarlar) --------
GBR_MAX_FEATURES = [None, "sqrt", "log2", 0.5, 0.7, 1.0]


def _gbr_loss_secenekleri():
    """Kurulu sklearn surumune uygun kare hata loss adini belirler."""
    X_k = np.arange(20, dtype=float).reshape(-1, 2)
    y_k = np.arange(10, dtype=float)
    for isim in ["squared_error", "ls"]:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                GradientBoostingRegressor(loss=isim, n_estimators=2).fit(X_k, y_k)
            return [isim, "huber"]
        except Exception:
            continue
    raise RuntimeError("Uygun GBR loss adi bulunamadi.")


GBR_LOSS = _gbr_loss_secenekleri()

PSO_UZAYI = {
    "SVR": {"bounds": [(-1.0, 4.0), (-5.0, 1.0),
                       (float(np.log10(0.001)), float(np.log10(0.30)))],
            "coz": lambda p: {"model__C": float(10.0 ** p[0]),
                              "model__gamma": float(10.0 ** p[1]),
                              "model__epsilon": float(10.0 ** p[2])}},
    "GBR": {"bounds": [(100.0, 1000.0),
                       (float(np.log10(0.005)), float(np.log10(0.20))),
                       (2.0, 5.0), (2.0, 20.0), (1.0, 10.0), (0.60, 1.00),
                       (0.0, float(len(GBR_MAX_FEATURES) - 1)),
                       (0.0, float(len(GBR_LOSS) - 1))],
            "coz": lambda p: {
                "model__n_estimators": int(np.clip(round(p[0]), 100, 1000)),
                "model__learning_rate": float(10.0 ** p[1]),
                "model__max_depth": int(np.clip(round(p[2]), 2, 5)),
                "model__min_samples_split": int(np.clip(round(p[3]), 2, 20)),
                "model__min_samples_leaf": int(np.clip(round(p[4]), 1, 10)),
                "model__subsample": float(np.clip(p[5], 0.60, 1.00)),
                "model__max_features": GBR_MAX_FEATURES[
                    int(np.clip(round(p[6]), 0, len(GBR_MAX_FEATURES) - 1))],
                "model__loss": GBR_LOSS[int(np.clip(round(p[7]), 0, len(GBR_LOSS) - 1))]}},
}


def pso_seed42_yeniden_calistir(model_adi, hedef):
    """Yalnizca geri kazanim basarisiz olursa kullanilir. Asama 5 ile ayni algoritma."""
    uzay = PSO_UZAYI[model_adi]
    sinirlar = np.array(uzay["bounds"], dtype=float)
    alt, ust = sinirlar[:, 0], sinirlar[:, 1]
    rng = np.random.default_rng(MAIN_PSO_SEED)
    hiz_siniri = PSO_VELOCITY_RATIO * (ust - alt)

    konum = rng.uniform(alt, ust, size=(PSO_SWARM_SIZE, len(alt)))
    hiz = rng.uniform(-hiz_siniri, hiz_siniri, size=konum.shape)
    pbest_konum, pbest_deger = konum.copy(), np.full(PSO_SWARM_SIZE, np.inf)
    gbest_konum, gbest_deger = konum[0].copy(), np.inf
    onbellek, sayac = {}, {"calls": 0, "unique": 0, "cached": 0}

    def degerlendir():
        nonlocal gbest_deger, gbest_konum
        for p in range(PSO_SWARM_SIZE):
            parametreler = uzay["coz"](konum[p])
            anahtar = tuple(sorted((k, str(v)) for k, v in parametreler.items()))
            sayac["calls"] += 1
            if anahtar in onbellek:
                sayac["cached"] += 1
                deger = onbellek[anahtar]
            else:
                deger = ortalama_cv_rmse(model_adi, hedef, parametreler)
                onbellek[anahtar] = deger
                sayac["unique"] += 1
            if deger < pbest_deger[p]:
                pbest_deger[p], pbest_konum[p] = deger, konum[p].copy()
            if deger < gbest_deger:
                gbest_deger, gbest_konum = deger, konum[p].copy()

    baslangic = time.perf_counter()
    degerlendir()
    guncelleme = max((PSO_N_ITERATIONS - 1) if PSO_COUNT_INITIAL_AS_ITERATION
                     else PSO_N_ITERATIONS, 0)
    for it in range(1, guncelleme + 1):
        w = PSO_W_START - (PSO_W_START - PSO_W_END) * (it - 1) / max(guncelleme - 1, 1)
        r1 = rng.random(konum.shape)
        r2 = rng.random(konum.shape)
        hiz = np.clip(w * hiz + PSO_C1 * r1 * (pbest_konum - konum)
                      + PSO_C2 * r2 * (gbest_konum - konum), -hiz_siniri, hiz_siniri)
        konum = np.clip(konum + hiz, alt, ust)
        degerlendir()
    sure = time.perf_counter() - baslangic

    return uzay["coz"](gbest_konum), float(gbest_deger), sayac, float(sure)


# ---- Eksik kombinasyonlarin yeniden calistirilmasi --------------------
YENIDEN_CALISTIRILAN = []
if eksik_kombinasyonlar:
    print("\nUYARI: Asagidaki kombinasyonlarda seed 42 hiperparametreleri "
          "mevcut kayitlardan geri kazanilamadi:")
    for model_adi, hedef in eksik_kombinasyonlar:
        print(f"  - {TARGET_KISA[hedef]} | {model_adi}")
    if not ALLOW_PSO_RERUN_IF_MISSING:
        raise RuntimeError("Geri kazanim basarisiz ve yeniden kosu kapali.")
    for model_adi, hedef in eksik_kombinasyonlar:
        print(f"  Yeniden calistiriliyor: {TARGET_KISA[hedef]} | {model_adi} (seed 42)")
        parametreler, deger, sayac, sure = pso_seed42_yeniden_calistir(model_adi, hedef)
        pso_en_iyi[(model_adi, hedef)] = parametreleri_duzelt(parametreler)
        YENIDEN_CALISTIRILAN.append({"Target": TARGET_KISA[hedef], "Model": model_adi,
                                     "Best_CV_RMSE": deger,
                                     "Total_objective_calls": sayac["calls"],
                                     "Unique_evaluations": sayac["unique"],
                                     "Cached_evaluations": sayac["cached"],
                                     "Runtime_seconds": sure})
else:
    print("\nTum seed 42 hiperparametreleri mevcut kayitlardan geri kazanildi. "
          "PSO YENIDEN CALISTIRILMADI.")

# ---- Geri kazanilan parametrelerin sayisal dogrulanmasi ---------------
print("\n--- Seed 42 CV RMSE dogrulamasi (kayit vs yeniden hesap) ---")
dogrulama_satirlari = []
for hedef in TARGETS:
    for model_adi in MODELLER:
        kisa = TARGET_KISA[hedef]
        parametreler = pso_en_iyi[(model_adi, hedef)]
        yeniden = ortalama_cv_rmse(model_adi, hedef, parametreler)

        kayitli = np.nan
        if not seed42_kosular.empty:
            eslesen = seed42_kosular[(seed42_kosular["Model"] == model_adi) &
                                     (seed42_kosular["Target"] == kisa)]
            if not eslesen.empty:
                kayitli = float(eslesen["Best_CV_RMSE"].iloc[0])
        capraz = pso_parcacik_rmse.get((model_adi, hedef), np.nan)

        fark = abs(yeniden - kayitli) if not np.isnan(kayitli) else np.nan
        durum = "OK" if (not np.isnan(fark) and fark < 1e-6) else \
                ("KONTROL" if not np.isnan(fark) else "KAYIT YOK")
        dogrulama_satirlari.append({
            "Target": kisa, "Model": model_adi,
            "Recorded_best_CV_RMSE_seed42": kayitli,
            "Particle_file_CV_RMSE": capraz,
            "Recomputed_CV_RMSE": yeniden,
            "Absolute_difference": fark, "Status": durum})
        print(f"  {kisa} | {model_adi:3s} | kayit={kayitli:.5f} | "
              f"yeniden={yeniden:.5f} | fark={fark:.2e} | {durum}")

dogrulama_df = pd.DataFrame(dogrulama_satirlari)
if (dogrulama_df["Status"] == "KONTROL").any():
    print("UYARI: Bazi kombinasyonlarda kayitli ve yeniden hesaplanan CV RMSE "
          "birebir ortusmuyor. Ayrintilar Optimized_Model_Parameters.xlsx icinde.")

In [ ]:
# =====================================================================
# BOLUM 3 - Baseline / Randomized Search / PSO(seed 42) icin
#           CV, train-test, tahmin ve alt grup sonuclarinin yeniden uretimi
# =====================================================================

def yontem_parametreleri(yontem, model_adi, hedef):
    """Ilgili yontemin hiperparametrelerini dondurur (PSO artik yalnizca seed 42)."""
    if yontem == "Baseline":
        return baseline_parametreleri(model_adi)
    if yontem == "Randomized Search":
        return rs_en_iyi[(model_adi, hedef)]
    if yontem == "PSO":
        return pso_en_iyi[(model_adi, hedef)]
    raise ValueError(f"Bilinmeyen yontem: {yontem}")


def metrik_seti(gercek, tahmin):
    """R2, RMSE, MAE ve Mean Error dondurur."""
    gercek = np.asarray(gercek, dtype=float)
    tahmin = np.asarray(tahmin, dtype=float)
    return {"R2": r2_score(gercek, tahmin),
            "RMSE": float(np.sqrt(mean_squared_error(gercek, tahmin))),
            "MAE": mean_absolute_error(gercek, tahmin),
            "Mean_Error": float(np.mean(tahmin - gercek))}


def guvenli_yuzde_hata(gercek, tahmin, eps=EPS):
    """Sifira bolunmeye karsi guvenli yuzde hata (%)."""
    gercek = np.asarray(gercek, dtype=float)
    payda = np.where(np.abs(gercek) < eps, np.nan, gercek)
    return 100.0 * (np.asarray(tahmin, dtype=float) - gercek) / payda


def manuel_cv_metrikleri(model_adi, hedef, parametreler):
    """Ayni bes sabit fold uzerinde R2, RMSE, MAE hesaplar."""
    satirlar = []
    for fold, (tr, va) in enumerate(CV_SPLITS, start=1):
        pipe = pipeline_olustur(model_adi)
        if parametreler:
            pipe.set_params(**parametreler)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            pipe.fit(X_train.iloc[tr], y_train[hedef].iloc[tr])
        tahmin = pipe.predict(X_train.iloc[va])
        gercek = y_train[hedef].iloc[va]
        satirlar.append({"Fold": fold, "R2": r2_score(gercek, tahmin),
                         "RMSE": float(np.sqrt(mean_squared_error(gercek, tahmin))),
                         "MAE": mean_absolute_error(gercek, tahmin)})
    tablo = pd.DataFrame(satirlar)
    ozet = {"CV_R2_mean": tablo["R2"].mean(), "CV_R2_std": tablo["R2"].std(),
            "CV_RMSE_mean": tablo["RMSE"].mean(), "CV_RMSE_std": tablo["RMSE"].std(),
            "CV_MAE_mean": tablo["MAE"].mean(), "CV_MAE_std": tablo["MAE"].std()}
    return ozet, tablo


cv_ozet_satirlari, cv_fold_satirlari = [], []
tt_satirlari, tahmin_satirlari = [], []
altgrup_satirlari, yuksek_frekans_satirlari = [], []
egitilmis_modeller = {}

print("\n--- Uc yontemin yeniden degerlendirilmesi ---")
for hedef in TARGETS:
    for model_adi in MODELLER:
        for yontem in YONTEMLER:
            kisa = TARGET_KISA[hedef]
            parametreler = yontem_parametreleri(yontem, model_adi, hedef)

            # (a) Manuel 5 katli GroupKFold metrikleri
            ozet, fold_tablosu = manuel_cv_metrikleri(model_adi, hedef, parametreler)
            satir = {"Target": kisa, "Model": model_adi, "Optimization_method": yontem}
            satir.update(ozet)
            satir["PSO_seed"] = MAIN_PSO_SEED if yontem == "PSO" else np.nan
            cv_ozet_satirlari.append(satir)
            cv_fold_satirlari.append(fold_tablosu.assign(
                Target=kisa, Model=model_adi, Optimization_method=yontem))

            # (b) Tum egitim kumesinde egitim ve bagimsiz test
            pipe = pipeline_olustur(model_adi)
            if parametreler:
                pipe.set_params(**parametreler)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                pipe.fit(X_train, y_train[hedef])
            egitilmis_modeller[(yontem, model_adi, hedef)] = pipe

            tahmin_tr = pipe.predict(X_train)
            tahmin_te = pipe.predict(X_test)
            m_tr = metrik_seti(y_train[hedef], tahmin_tr)
            m_te = metrik_seti(y_test[hedef], tahmin_te)

            tt_satirlari.append({
                "Target": kisa, "Model": model_adi, "Optimization_method": yontem,
                "PSO_seed": MAIN_PSO_SEED if yontem == "PSO" else np.nan,
                "Best_parameters": str(parametreler),
                "Train_R2": m_tr["R2"], "Train_RMSE": m_tr["RMSE"], "Train_MAE": m_tr["MAE"],
                "Test_R2": m_te["R2"], "Test_RMSE": m_te["RMSE"], "Test_MAE": m_te["MAE"],
                "Test_Mean_Error": m_te["Mean_Error"],
                "R2_gap_train_minus_test": m_tr["R2"] - m_te["R2"]})

            # (c) Test tahminleri
            gercek = y_test[hedef].values
            tahmin_satirlari.append(pd.DataFrame({
                "Geometry_ID": geo_test.values,
                "Optimization_Method": yontem,
                "PSO_seed": MAIN_PSO_SEED if yontem == "PSO" else np.nan,
                "Model": model_adi, "Target": kisa,
                "Actual": gercek, "Predicted": tahmin_te,
                "Residual": gercek - tahmin_te,
                "Signed_Error": tahmin_te - gercek,
                "Absolute_Error": np.abs(tahmin_te - gercek),
                "Percentage_Error": guvenli_yuzde_hata(gercek, tahmin_te),
                "Outside_Training_Range": np.where(disarida_mask, "Yes", "No"),
                "High_Frequency_Group": np.where(
                    yuksek_frekans_mask[hedef], f"> {HIGH_FREQ_THRESHOLD} Hz",
                    f"<= {HIGH_FREQ_THRESHOLD} Hz")}))

            # (d) Tasarim uzayi ici/disi alt gruplari
            for etiket, maske in [("Inside training range", ~disarida_mask),
                                  ("Outside training range", disarida_mask)]:
                kayit = {"Target": kisa, "Model": model_adi, "Optimization_method": yontem,
                         "Subgroup": etiket, "N_records": int(maske.sum())}
                if maske.sum() > 1:
                    m = metrik_seti(gercek[maske], tahmin_te[maske])
                    kayit.update({"RMSE": m["RMSE"], "MAE": m["MAE"],
                                  "Mean_Error": m["Mean_Error"]})
                else:
                    kayit.update({"RMSE": np.nan, "MAE": np.nan, "Mean_Error": np.nan})
                altgrup_satirlari.append(kayit)

            # (e) Yuksek frekans alt gruplari (yalnizca tanisal)
            for etiket, maske in [(f"<= {HIGH_FREQ_THRESHOLD} Hz", ~yuksek_frekans_mask[hedef]),
                                  (f"> {HIGH_FREQ_THRESHOLD} Hz", yuksek_frekans_mask[hedef])]:
                kayit = {"Target": kisa, "Model": model_adi, "Optimization_method": yontem,
                         "Frequency_group": etiket, "N_records": int(maske.sum())}
                if maske.sum() > 0:
                    m = metrik_seti(gercek[maske], tahmin_te[maske]) if maske.sum() > 1 else \
                        {"RMSE": float(abs(tahmin_te[maske][0] - gercek[maske][0])),
                         "MAE": float(abs(tahmin_te[maske][0] - gercek[maske][0])),
                         "Mean_Error": float(tahmin_te[maske][0] - gercek[maske][0])}
                    kayit.update({"RMSE": m["RMSE"], "MAE": m["MAE"],
                                  "Mean_Error": m["Mean_Error"]})
                else:
                    kayit.update({"RMSE": np.nan, "MAE": np.nan, "Mean_Error": np.nan})
                yuksek_frekans_satirlari.append(kayit)

        print(f"  {TARGET_KISA[hedef]} | {model_adi:3s} | uc yontem tamamlandi")

cv_ozet_df = pd.DataFrame(cv_ozet_satirlari)
cv_fold_df = pd.concat(cv_fold_satirlari, ignore_index=True)
tt_df = pd.DataFrame(tt_satirlari)
tahmin_df = pd.concat(tahmin_satirlari, ignore_index=True)
altgrup_df = pd.DataFrame(altgrup_satirlari)
yuksek_frekans_df = pd.DataFrame(yuksek_frekans_satirlari)

In [ ]:
# =====================================================================
# BOLUM 4 - PSO joblib modellerinin seed 42 ile YENIDEN kaydedilmesi
# Randomized Search modelleri degismedigi icin dogrulanir, uzerine yazilmaz.
# =====================================================================

kaydedilen_modeller = []
for hedef in TARGETS:
    for model_adi in MODELLER:
        kisa = TARGET_KISA[hedef]
        dosya = f"PSO_{model_adi}_{kisa}.joblib"
        joblib.dump(egitilmis_modeller[("PSO", model_adi, hedef)],
                    os.path.join(MODEL_DIR, dosya))
        kaydedilen_modeller.append({"File": dosya, "Method": "PSO",
                                    "Seed": MAIN_PSO_SEED, "Status": "Overwritten (seed 42)"})
        print("Yeniden kaydedildi:", dosya)

        rs_dosya = f"RandomSearch_{model_adi}_{kisa}.joblib"
        rs_yol = os.path.join(MODEL_DIR, rs_dosya)
        if os.path.exists(rs_yol):
            durum = "Unchanged (verified)"
        else:
            joblib.dump(egitilmis_modeller[("Randomized Search", model_adi, hedef)], rs_yol)
            durum = "Recreated (was missing)"
        kaydedilen_modeller.append({"File": rs_dosya, "Method": "Randomized Search",
                                    "Seed": np.nan, "Status": durum})

model_durum_df = pd.DataFrame(kaydedilen_modeller)

In [ ]:
# =====================================================================
# BOLUM 5 - Esit butce karsilastirmasi ve hesaplama maliyeti
# PSO kararlilik kosularinin maliyeti ANA karsilastirmaya DAHIL EDILMEZ.
# =====================================================================

def gercek_rs_butcesi(model_adi, hedef):
    """Randomized Search icin gercek aday ve fit sayisini kayitlardan okur."""
    sayfa = f"{TARGET_KISA[hedef]}_{model_adi}"
    try:
        tablo = pd.read_excel(RS_SONUC_FILE, sheet_name=sayfa)
        aday = int(tablo["Total_candidate_count"].iloc[0]) if "Total_candidate_count" \
            in tablo.columns else int(len(tablo))
        fit = int(tablo["Total_fit_count"].iloc[0]) if "Total_fit_count" in tablo.columns \
            else aday * N_SPLITS
        return aday, aday, 0, fit
    except Exception:
        return np.nan, np.nan, np.nan, np.nan


def gercek_pso_butcesi(model_adi, hedef):
    """PSO seed 42 icin gercek cagri, benzersiz degerlendirme ve fit sayisini okur."""
    yeniden = [r for r in YENIDEN_CALISTIRILAN
               if r["Model"] == model_adi and r["Target"] == TARGET_KISA[hedef]]
    if yeniden:
        r = yeniden[0]
        return (r["Total_objective_calls"], r["Unique_evaluations"],
                r["Cached_evaluations"], r["Unique_evaluations"] * N_SPLITS)
    if not seed42_kosular.empty:
        eslesen = seed42_kosular[(seed42_kosular["Model"] == model_adi) &
                                 (seed42_kosular["Target"] == TARGET_KISA[hedef])]
        if not eslesen.empty:
            satir = eslesen.iloc[0]
            benzersiz = int(satir.get("Unique_evaluations", np.nan))
            return (int(satir.get("Total_objective_calls", np.nan)), benzersiz,
                    int(satir.get("Cached_evaluations", 0)), benzersiz * N_SPLITS)
    return np.nan, np.nan, np.nan, np.nan


def eski_sureyi_oku(model_adi, hedef, yontem):
    """Onceki maliyet dosyasindan calisma suresini alir (varsa)."""
    try:
        eski = pd.read_excel(ESKI_COST_FILE, sheet_name="Cost_comparison")
        eslesen = eski[(eski["Model"] == model_adi) &
                       (eski["Target"] == TARGET_KISA[hedef]) &
                       (eski["Optimization_method"] == yontem)]
        return float(eslesen["Runtime_seconds"].iloc[0]) if not eslesen.empty else np.nan
    except Exception:
        return np.nan


def pso_seed42_suresi(model_adi, hedef):
    """Seed 42 koşusunun suresi kararlilik dosyasindan okunur."""
    yeniden = [r for r in YENIDEN_CALISTIRILAN
               if r["Model"] == model_adi and r["Target"] == TARGET_KISA[hedef]]
    if yeniden:
        return yeniden[0]["Runtime_seconds"]
    if not seed42_kosular.empty:
        eslesen = seed42_kosular[(seed42_kosular["Model"] == model_adi) &
                                 (seed42_kosular["Target"] == TARGET_KISA[hedef])]
        if not eslesen.empty:
            return float(eslesen["Runtime_seconds"].iloc[0])
    return np.nan


# ---- Ana karsilastirma tablosu (butce sutunlariyla) -------------------
karsilastirma_satirlari = []
for hedef in TARGETS:
    for model_adi in MODELLER:
        kisa = TARGET_KISA[hedef]
        for yontem in YONTEMLER:
            cv = cv_ozet_df[(cv_ozet_df["Target"] == kisa) &
                            (cv_ozet_df["Model"] == model_adi) &
                            (cv_ozet_df["Optimization_method"] == yontem)].iloc[0]
            tt = tt_df[(tt_df["Target"] == kisa) & (tt_df["Model"] == model_adi) &
                       (tt_df["Optimization_method"] == yontem)].iloc[0]

            if yontem == "Randomized Search":
                cagri, benzersiz, onbellek, fit = gercek_rs_butcesi(model_adi, hedef)
                sure = eski_sureyi_oku(model_adi, hedef, yontem)
            elif yontem == "PSO":
                cagri, benzersiz, onbellek, fit = gercek_pso_butcesi(model_adi, hedef)
                sure = pso_seed42_suresi(model_adi, hedef)
            else:
                cagri = benzersiz = onbellek = fit = 0
                sure = np.nan

            karsilastirma_satirlari.append({
                "Target": kisa, "Model": model_adi, "Optimization_method": yontem,
                "PSO_seed": MAIN_PSO_SEED if yontem == "PSO" else np.nan,
                "Candidate_evaluations": cagri,
                "Unique_candidate_evaluations": benzersiz,
                "Cached_evaluations": onbellek,
                "Model_fits": fit,
                "Runtime_seconds": sure,
                "CV_R2_mean": cv["CV_R2_mean"], "CV_R2_std": cv["CV_R2_std"],
                "CV_RMSE_mean": cv["CV_RMSE_mean"], "CV_RMSE_std": cv["CV_RMSE_std"],
                "CV_MAE_mean": cv["CV_MAE_mean"], "CV_MAE_std": cv["CV_MAE_std"],
                "Train_R2": tt["Train_R2"], "Train_RMSE": tt["Train_RMSE"],
                "Train_MAE": tt["Train_MAE"],
                "Test_R2": tt["Test_R2"], "Test_RMSE": tt["Test_RMSE"],
                "Test_MAE": tt["Test_MAE"],
                "Best_parameters": tt["Best_parameters"]})

karsilastirma_df = pd.DataFrame(karsilastirma_satirlari)


def iyilesme_yuzdesi(referans, yeni):
    """(referans - yeni)/referans*100. Negatif deger kotulesmedir."""
    if referans is None or np.isnan(referans) or abs(referans) < EPS:
        return np.nan
    return 100.0 * (referans - yeni) / referans


iyilesme_satirlari = []
for hedef in TARGETS:
    for model_adi in MODELLER:
        kisa = TARGET_KISA[hedef]
        alt = karsilastirma_df[(karsilastirma_df["Target"] == kisa) &
                               (karsilastirma_df["Model"] == model_adi)]
        al = lambda y, s: alt[alt["Optimization_method"] == y][s].iloc[0]
        for kaynak, varis in [("Baseline", "Randomized Search"), ("Baseline", "PSO"),
                              ("Randomized Search", "PSO")]:
            iyilesme_satirlari.append({
                "Target": kisa, "Model": model_adi,
                "Comparison": f"{kaynak} -> {varis}",
                "PSO_seed": MAIN_PSO_SEED,
                "CV_RMSE_from": al(kaynak, "CV_RMSE_mean"),
                "CV_RMSE_to": al(varis, "CV_RMSE_mean"),
                "CV_RMSE_improvement_%": iyilesme_yuzdesi(al(kaynak, "CV_RMSE_mean"),
                                                          al(varis, "CV_RMSE_mean")),
                "Test_RMSE_from": al(kaynak, "Test_RMSE"),
                "Test_RMSE_to": al(varis, "Test_RMSE"),
                "Test_RMSE_improvement_%": iyilesme_yuzdesi(al(kaynak, "Test_RMSE"),
                                                            al(varis, "Test_RMSE"))})
iyilesme_df = pd.DataFrame(iyilesme_satirlari)

# ---- Kararlilik kosulari: yalnizca AYRI maliyet ozeti -----------------
kararlilik_maliyet = pd.DataFrame()
if os.path.exists(PSO_KARAR_FILE):
    tum_kosular = pd.read_excel(PSO_KARAR_FILE, sheet_name="Runs")
    kararlilik_maliyet = (tum_kosular.groupby(["Target", "Model"])
                          .agg(Number_of_runs=("Seed", "count"),
                               Total_objective_calls_all_runs=("Total_objective_calls", "sum"),
                               Total_unique_evaluations_all_runs=("Unique_evaluations", "sum"),
                               Total_runtime_seconds_all_runs=("Runtime_seconds", "sum"))
                          .reset_index())
    kararlilik_maliyet["Total_model_fits_all_runs"] = \
        kararlilik_maliyet["Total_unique_evaluations_all_runs"] * N_SPLITS
    kararlilik_maliyet["Note"] = ("Stability analysis only. NOT included in the "
                                  "equal-budget comparison.")

# ---- Excel dosyalarinin guncellenmesi ---------------------------------
GUNCELLENEN_DOSYALAR = []


def excel_yaz(dosya_adi, yazma_fonksiyonu):
    """Excel dosyasini yazar ve guncellenen dosya listesine ekler."""
    yol = os.path.join(OUTPUT_DIR, dosya_adi)
    with pd.ExcelWriter(yol, engine="openpyxl") as writer:
        yazma_fonksiyonu(writer)
    GUNCELLENEN_DOSYALAR.append(dosya_adi)
    print("Guncellendi:", dosya_adi)


def _params_yaz(writer):
    tt_df[["Target", "Model", "Optimization_method", "PSO_seed",
           "Best_parameters"]].to_excel(writer, sheet_name="Best_parameters", index=False)
    pd.DataFrame(geri_kazanim_notlari).to_excel(writer, sheet_name="Recovery_status", index=False)
    dogrulama_df.round(8).to_excel(writer, sheet_name="Seed42_verification", index=False)
    if YENIDEN_CALISTIRILAN:
        pd.DataFrame(YENIDEN_CALISTIRILAN).round(6).to_excel(
            writer, sheet_name="Reruns", index=False)
    model_durum_df.to_excel(writer, sheet_name="Saved_models", index=False)


def _cv_yaz(writer):
    cv_ozet_df.round(6).to_excel(writer, sheet_name="CV_summary", index=False)
    for hedef in TARGETS:
        for model_adi in MODELLER:
            alt = cv_fold_df[(cv_fold_df["Target"] == TARGET_KISA[hedef]) &
                             (cv_fold_df["Model"] == model_adi)]
            alt.round(6).to_excel(writer,
                                  sheet_name=f"folds_{TARGET_KISA[hedef]}_{model_adi}",
                                  index=False)


def _tt_yaz(writer):
    for hedef in TARGETS:
        tt_df[tt_df["Target"] == TARGET_KISA[hedef]].round(6).to_excel(
            writer, sheet_name=f"{TARGET_KISA[hedef]}_train_test", index=False)


def _karsilastirma_yaz(writer):
    for hedef in TARGETS:
        karsilastirma_df[karsilastirma_df["Target"] == TARGET_KISA[hedef]].round(6).to_excel(
            writer, sheet_name=f"{TARGET_KISA[hedef]}_comparison", index=False)
    iyilesme_df.round(6).to_excel(writer, sheet_name="Improvements", index=False)
    pd.DataFrame([
        {"Note": "The PSO results in this file correspond to the seed 42 run only."},
        {"Note": "Randomized Search and PSO use the same evaluation budget and the same "
                 "five fixed GroupKFold folds."},
        {"Note": "The five-seed PSO runs are reported separately as a stability analysis "
                 "and are excluded from this comparison."},
    ]).to_excel(writer, sheet_name="Notes", index=False)


def _maliyet_yaz(writer):
    karsilastirma_df[karsilastirma_df["Optimization_method"] != "Baseline"][
        ["Target", "Model", "Optimization_method", "PSO_seed", "Candidate_evaluations",
         "Unique_candidate_evaluations", "Cached_evaluations", "Model_fits",
         "Runtime_seconds", "CV_RMSE_mean", "CV_RMSE_std", "Test_RMSE", "Test_MAE",
         "Test_R2"]].round(6).to_excel(writer, sheet_name="Equal_budget_comparison", index=False)
    if not kararlilik_maliyet.empty:
        kararlilik_maliyet.round(6).to_excel(
            writer, sheet_name="PSO_stability_cost", index=False)
    CV_KONTROL.to_excel(writer, sheet_name="CV_fold_check", index=False)
    pd.DataFrame([
        {"Setting": "Main PSO seed", "Value": MAIN_PSO_SEED},
        {"Setting": "PSO best-of-seeds selection", "Value": "Disabled (corrected)"},
        {"Setting": "Hyperparameter ranges", "Value": "Unchanged from Stage 5"},
        {"Setting": "Optimization re-executed",
         "Value": "Yes (partial)" if YENIDEN_CALISTIRILAN else "No"},
        {"Setting": "Runtime comparability",
         "Value": "Runtime depends on visited hyperparameters; use model fits as the "
                  "fair cost metric."},
    ]).to_excel(writer, sheet_name="Settings", index=False)


def _tahmin_yaz(writer):
    for hedef in TARGETS:
        for model_adi in MODELLER:
            alt = tahmin_df[(tahmin_df["Target"] == TARGET_KISA[hedef]) &
                            (tahmin_df["Model"] == model_adi)]
            alt.round(6).to_excel(writer,
                                  sheet_name=f"{TARGET_KISA[hedef]}_{model_adi}", index=False)


def _altgrup_yaz(writer):
    for hedef in TARGETS:
        altgrup_df[altgrup_df["Target"] == TARGET_KISA[hedef]].round(6).to_excel(
            writer, sheet_name=f"{TARGET_KISA[hedef]}_subgroups", index=False)
    pd.DataFrame({"Outside_range_geometries": sorted(bulunan_disarida)}).to_excel(
        writer, sheet_name="Outside_geometries", index=False)


def _yuksek_frekans_yaz(writer):
    for hedef in TARGETS:
        yuksek_frekans_df[yuksek_frekans_df["Target"] == TARGET_KISA[hedef]].round(6).to_excel(
            writer, sheet_name=f"{TARGET_KISA[hedef]}_high_freq", index=False)
    pd.DataFrame([{"Note": "The 2.5 Hz threshold is a diagnostic grouping only. It was not "
                           "used in hyperparameter selection, optimization, fitness "
                           "weighting or sample weighting."}]).to_excel(
        writer, sheet_name="Notes", index=False)


excel_yaz("Optimized_Model_Parameters.xlsx", _params_yaz)
excel_yaz("Optimized_CV_Results.xlsx", _cv_yaz)
excel_yaz("Optimized_Train_Test_Results.xlsx", _tt_yaz)
excel_yaz("Baseline_RandomSearch_PSO_Comparison.xlsx", _karsilastirma_yaz)
excel_yaz("Optimization_Computational_Cost.xlsx", _maliyet_yaz)
excel_yaz("Optimized_Test_Predictions.xlsx", _tahmin_yaz)
excel_yaz("Optimized_Range_Subgroup_Results.xlsx", _altgrup_yaz)
excel_yaz("Optimized_HighFrequency_Results.xlsx", _yuksek_frekans_yaz)

In [ ]:
# =====================================================================
# BOLUM 6 - Actual-vs-predicted, residual ve yontem karsilastirma
#           grafiklerinin seed 42 sonuclariyla yeniden uretimi
# =====================================================================

METRIK_ETIKET = {"CV_RMSE": "Cross-validation RMSE (Hz)", "Test_RMSE": "Test RMSE (Hz)",
                 "Test_MAE": "Test MAE (Hz)", "Test_R2": "Test $R^2$ (-)"}


def yontem_karsilastirma_grafigi(hedef_kisa, model_adi, metrik):
    """Baseline / Randomized Search / PSO(seed 42) karsilastirma cubuk grafigi."""
    alt = karsilastirma_df[(karsilastirma_df["Target"] == hedef_kisa) &
                           (karsilastirma_df["Model"] == model_adi)]
    alt = alt.set_index("Optimization_method").loc[YONTEMLER].reset_index()
    if metrik == "CV_RMSE":
        deger, hata = alt["CV_RMSE_mean"].values, alt["CV_RMSE_std"].values
    else:
        deger, hata = alt[metrik].values, None

    fig, ax = plt.subplots(figsize=(5.6, 4.2))
    ax.bar(alt["Optimization_method"], deger, yerr=hata, capsize=4,
           color=[RENK[y] for y in alt["Optimization_method"]],
           edgecolor="white", width=0.6,
           error_kw=dict(ecolor="#333333", lw=1.0))
    ax.set_ylabel(METRIK_ETIKET[metrik])
    ax.set_xlabel("Optimization method")
    ax.set_title(f"{model_adi} - Target: {HEDEF_ETIKET[hedef_kisa]}", loc="left")
    sns.despine(ax=ax)
    fig.tight_layout()
    kaydet(fig, COMP_DIR, f"Fig_{metrik}_{hedef_kisa}_{model_adi}")


def avp_grafigi(yontem, model_adi, hedef):
    """Optimize edilmis model icin actual vs predicted grafigi."""
    kisa = TARGET_KISA[hedef]
    gercek = y_test[hedef].values
    tahmin = egitilmis_modeller[(yontem, model_adi, hedef)].predict(X_test)
    m = metrik_seti(gercek, tahmin)

    fig, ax = plt.subplots(figsize=(5.4, 5.2))
    ic = ~disarida_mask
    ax.scatter(gercek[ic], tahmin[ic], s=26, color="#4C72B0", alpha=0.6,
               edgecolors="none", label="Inside training range")
    ax.scatter(gercek[disarida_mask], tahmin[disarida_mask], s=52, marker="^",
               facecolors="none", edgecolors=RENK_VURGU, linewidths=1.4,
               label="Outside training range")
    alt_s, ust_s = min(gercek.min(), tahmin.min()), max(gercek.max(), tahmin.max())
    pay = 0.05 * (ust_s - alt_s)
    ax.plot([alt_s - pay, ust_s + pay], [alt_s - pay, ust_s + pay],
            color="#333333", lw=1.2, ls="--", label="y = x")
    ax.set_xlim(alt_s - pay, ust_s + pay)
    ax.set_ylim(alt_s - pay, ust_s + pay)
    ax.set_aspect("equal", adjustable="box")
    ax.text(0.04, 0.96,
            f"$R^2$ = {m['R2']:.3f}\nRMSE = {m['RMSE']:.4f} Hz\n"
            f"MAE = {m['MAE']:.4f} Hz\nMean error = {m['Mean_Error']:.4f} Hz",
            transform=ax.transAxes, va="top", ha="left", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white",
                      edgecolor="0.8", alpha=0.9))
    baslik = f"{model_adi} - {yontem}" + (f" (seed {MAIN_PSO_SEED})" if yontem == "PSO" else "")
    ax.set_xlabel(f"Actual {HEDEF_ETIKET[kisa]} (Hz)")
    ax.set_ylabel(f"Predicted {HEDEF_ETIKET[kisa]} (Hz)")
    ax.set_title(baslik, loc="left")
    ax.legend(frameon=False, loc="lower right")
    sns.despine(ax=ax)
    fig.tight_layout()
    kaydet(fig, AVP_DIR, f"Fig_AVP_{kisa}_{model_adi}_{yontem.replace(' ', '')}")


def residual_grafigi(yontem, model_adi, hedef):
    """Predicted vs residual grafigi; alt gruplar farkli isaretleyicilerle."""
    kisa = TARGET_KISA[hedef]
    gercek = y_test[hedef].values
    tahmin = egitilmis_modeller[(yontem, model_adi, hedef)].predict(X_test)
    artik = gercek - tahmin
    yuksek = yuksek_frekans_mask[hedef]

    fig, ax = plt.subplots(figsize=(5.8, 4.6))
    normal = (~disarida_mask) & (~yuksek)
    ax.scatter(tahmin[normal], artik[normal], s=26, color="#4C72B0", alpha=0.6,
               edgecolors="none", label="Inside range, $\\leq$ 2.5 Hz")
    ax.scatter(tahmin[(~disarida_mask) & yuksek], artik[(~disarida_mask) & yuksek],
               s=44, marker="s", facecolors="none", edgecolors="#333333",
               linewidths=1.2, label="Inside range, > 2.5 Hz")
    ax.scatter(tahmin[disarida_mask], artik[disarida_mask], s=52, marker="^",
               facecolors="none", edgecolors=RENK_VURGU, linewidths=1.4,
               label="Outside training range")
    ax.axhline(0, color="#333333", lw=1.1, ls="--")
    baslik = f"{model_adi} - {yontem}" + (f" (seed {MAIN_PSO_SEED})" if yontem == "PSO" else "")
    ax.set_xlabel(f"Predicted {HEDEF_ETIKET[kisa]} (Hz)")
    ax.set_ylabel(f"Residual (actual - predicted) {HEDEF_ETIKET[kisa]} (Hz)")
    ax.set_title(baslik, loc="left")
    ax.legend(frameon=False, loc="best")
    sns.despine(ax=ax)
    fig.tight_layout()
    kaydet(fig, RESID_DIR, f"Fig_Residual_{kisa}_{model_adi}_{yontem.replace(' ', '')}")


print("\n--- Grafikler yeniden uretiliyor ---")
for hedef in TARGETS:
    kisa = TARGET_KISA[hedef]
    for model_adi in MODELLER:
        for metrik in ["CV_RMSE", "Test_RMSE", "Test_MAE", "Test_R2"]:
            yontem_karsilastirma_grafigi(kisa, model_adi, metrik)
        for yontem in ["Randomized Search", "PSO"]:
            avp_grafigi(yontem, model_adi, hedef)
            residual_grafigi(yontem, model_adi, hedef)

In [ ]:
# =====================================================================
# BOLUM 7 - Nihai duzeltme raporu
# =====================================================================
print("\n" + "=" * 70)
print("METODOLOJIK DUZELTME RAPORU")
print("=" * 70)

print("\n[1] GUNCELLENEN EXCEL DOSYALARI")
for dosya in GUNCELLENEN_DOSYALAR:
    print("   -", dosya)
print("   Degistirilmeyen dosyalar: Randomized_Search_Results.xlsx, "
      "PSO_Search_Results.xlsx,\n   PSO_Convergence_Results.xlsx, PSO_Stability_Results.xlsx")

print("\n[2] SEED 42 ILE YENIDEN KAYDEDILEN MODELLER")
print(model_durum_df.to_string(index=False))

print("\n[3] GERCEK ADAY DEGERLENDIRME VE MODEL FIT SAYILARI")
butce = karsilastirma_df[karsilastirma_df["Optimization_method"] != "Baseline"][
    ["Target", "Model", "Optimization_method", "PSO_seed", "Candidate_evaluations",
     "Unique_candidate_evaluations", "Cached_evaluations", "Model_fits"]]
print(butce.to_string(index=False))

print("\n[4] YENIDEN OPTIMIZASYON DURUMU")
if YENIDEN_CALISTIRILAN:
    print("   Kismi yeniden kosu YAPILDI (yalnizca geri kazanilamayan kombinasyonlar):")
    print(pd.DataFrame(YENIDEN_CALISTIRILAN).round(6).to_string(index=False))
else:
    print("   Hicbir optimizasyon yeniden calistirilmadi.")
    print("   Tum seed 42 hiperparametreleri mevcut kayitlardan geri kazanildi.")

print("\n[5] SEED 42 DOGRULAMASI")
print(dogrulama_df.round(8).to_string(index=False))

print("\n[6] PSO KARARLILIK KOSULARININ MALIYETI (ana karsilastirmaya DAHIL DEGIL)")
if not kararlilik_maliyet.empty:
    print(kararlilik_maliyet.round(2).to_string(index=False))

print("\n[7] ANA ESIT BUTCE KARSILASTIRMASI - CV ve TEST RMSE")
ozet = karsilastirma_df[["Target", "Model", "Optimization_method", "Candidate_evaluations",
                         "Model_fits", "CV_RMSE_mean", "Test_RMSE", "Test_R2"]]
print(ozet.round(5).to_string(index=False))

print("\nDuzeltme tamamlandi. Hiperparametre araliklari degistirilmedi. "
      "Nihai model secimi yapilmadi.")
print("Cikti klasoru:", OUTPUT_DIR)